# Cuaderno de Pruebas Masivas

Este notebook ejecuta múltiples simulaciones con diferentes configuraciones de:
- Algoritmos de búsqueda
- Funciones objetivo
- Número de agentes
- Número de indicios

## Descripción General

Este notebook realiza un **estudio comparativo exhaustivo** de diferentes algoritmos de búsqueda mediante la ejecución de miles de simulaciones automatizadas. El objetivo es evaluar el rendimiento de diversos algoritmos bajo múltiples configuraciones y condiciones aleatorias.

## Características de las Pruebas

### Aleatoriedad en Cada Iteración

**Aspecto crítico**: En cada una de las 1000 iteraciones por configuración, se generan aleatoriamente:

- **Posiciones de los indicios**: Coordenadas aleatorias dentro de los límites del mapa
- **Posiciones iniciales de los agentes**: Ubicaciones de partida aleatorias para cada agente
- **Semilla de simulación**: Cada iteración usa una semilla diferente (0-999) para garantizar variabilidad

Esto asegura que los resultados no dependan de configuraciones iniciales específicas y permite obtener estadísticas robustas sobre el rendimiento real de cada algoritmo.

### Dimensiones del Estudio

El notebook ejecuta simulaciones variando sistemáticamente:

#### 1. Mapas (2 mapas)
- Mapa mediano
- Mapa grande

#### 2. Algoritmos de búsqueda (7 tipos)
- **Determinísticos**: voraz-myope, voraz-heur, lawnmower, expanding_sq
- **Metaheurísticos**: ACO (Ant Colony Optimization), ABC (Artificial Bee Colony), BHA (Black Hole Algorithm)

#### 3. Funciones objetivo (4 variantes, solo para metaheurísticos)
- **ET**: Eficiencia Temporal
- **DTR**: Distancia Total Recorrida
- **MS**: Minimización de Solapamiento
- **ME**: Maximización de Exploración

#### 4. Número de agentes (3 configuraciones)
- 1, 5, 10 agentes

#### 5. Número de indicios (3 configuraciones)
- 1, 3, 10 indicios a localizar

### Volumen de Pruebas

- **Iteraciones por configuración**: 1000
  - Algoritmos determinísticos: 4 (alg) × 3 (ag) × 3 (ind) * 2 (map) = 72 configuraciones * 1000 iteraciones = 72,000 ejecuciones
  - Algoritmos metaheurísticos: 3 (alg) × 4 (FO) × 3 (ag) × 3(ind) * 2 (map) = 216 configuraciones * 1000 iteraciones = 216,000 ejecuciones
- **Total de simulaciones**: ~288,000 ejecuciones

### Sistema de Notificaciones

El notebook incluye integración con **Telegram** para monitorear el progreso en tiempo real, enviando notificaciones al completar cada configuración y al finalizar todas las simulaciones.

---

## Objetivo del Estudio

Obtener datos estadísticamente significativos sobre el rendimiento de cada algoritmo en diferentes escenarios, permitiendo:

- Comparar algoritmos determinísticos vs metaheurísticos
- Evaluar el impacto del número de agentes en la eficiencia
- Analizar la escalabilidad con diferentes cantidades de indicios
- Determinar qué funciones objetivo son más efectivas para cada contexto

In [ ]:
import json
import copy
import random
from pathlib import Path
import subprocess
from tqdm import tqdm
import requests
import os

## Configuración bot telegram

In [33]:
TELEGRAM_TOKEN = "8340887367:AAG-uOUAQ2oVMrfhjM4wLeXA8hHxCGIt26E"
CHAT_ID = "2114172581"

def enviar_mensaje_telegram(mensaje):
    """
    Envía un mensaje de notificación vía Telegram
    """
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    data = {"chat_id": CHAT_ID, "text": mensaje}
    try:
        requests.post(url, data=data)
    except Exception as e:
        print("Error enviando mensaje a Telegram:", e)

## Configuración inicial

In [ ]:
# Obtener el directorio donde está el script
SCRIPT_DIR = Path(__file__).parent.resolve()

# Crear carpetas necesarias (relativas al script)
CARPETA_JSON = SCRIPT_DIR / "json"
CARPETA_RESULTADOS = SCRIPT_DIR / "resultados_pruebas_masivas"

CARPETA_JSON.mkdir(exist_ok=True)
CARPETA_RESULTADOS.mkdir(exist_ok=True)

print(f"Carpetas creadas/verificadas:")
print(f"  - {CARPETA_JSON}")
print(f"  - {CARPETA_RESULTADOS}")

# Rutas de mapas (relativas al script)
mapa_mediano = SCRIPT_DIR / "pruebas" / "mediano-indicios-1-agentes-1-random.json"
mapa_grande = SCRIPT_DIR / "pruebas" / "grande-indicios-1-agentes-1-random.json"

# Verificar que existen los archivos
MAPAS = []
for mapa in [mapa_mediano, mapa_grande]:
    if mapa.exists():
        MAPAS.append(str(mapa))
        print(f"Encontrado: {mapa.name}")
    else:
        print(f"No encontrado: {mapa}")

if not MAPAS:
    print("\nERROR: No se encontraron archivos de mapas.")
    print(f"Directorio de búsqueda: {SCRIPT_DIR / 'pruebas'}")
    print(f"\nArchivos disponibles en pruebas/:")
    pruebas_dir = SCRIPT_DIR / "pruebas"
    if pruebas_dir.exists():
        for f in sorted(pruebas_dir.glob("*.json")):
            print(f"  - {f.name}")
    exit(1)

print(f"\nMapas a procesar: {len(MAPAS)}")

# Algoritmos a probar
ALGORITMOS = [
    "voraz-myope", "voraz-heur", "lawnmower", "expanding_sq",
    "ACO", "ABC", "BHA"
]

# Ruta del simulador (está en el directorio padre de TFG_Romeo)
alg = SCRIPT_DIR.parent / "bf-busqueda.py"

print(f"\nBuscando simulador en: {alg}")
if not alg.exists():
    print(f"✗ No se encontró el simulador")
    # Intentar buscar en la misma carpeta
    alg_local = SCRIPT_DIR / "bf-busqueda.py"
    if alg_local.exists():
        alg = alg_local
        print(f"Encontrado en: {alg}")
    else:
        print("\nArchivos .py disponibles en el directorio padre:")
        for f in sorted(SCRIPT_DIR.parent.glob("*.py")):
            print(f"  - {f.name}")
        alg_input = input("\nIngresa la ruta correcta al simulador bf-busqueda.py: ")
        alg = Path(alg_input)
else:
    print(f"Simulador encontrado")

alg = str(alg)

# Funciones objetivo (solo para ACO, ABC, BHA)
FUNCIONES_OBJ = ["ET", "DTR", "MS", "ME"]

# Variaciones de agentes e indicios
NUM_AGENTES = [1, 3, 10]
NUM_INDICIOS = [1, 3, 10]

# Número de iteraciones por configuración
N_ITERACIONES = 1000

print("\n" + "="*60)
print("CONFIGURACIÓN CARGADA")
print("="*60)
print(f"  - Mapas: {len(MAPAS)}")
print(f"  - Algoritmos: {len(ALGORITMOS)}")
print(f"  - Iteraciones por configuración: {N_ITERACIONES}")
print(f"  - Carpeta JSON: {CARPETA_JSON}")
print(f"  - Carpeta resultados: {CARPETA_RESULTADOS}")
print("="*60 + "\n")

Configuración cargada
  - Mapas: 2
  - Algoritmos: 7
  - Iteraciones por configuración: 1000


## Funciones auxiliares

In [ ]:
def modificar_config(base_config, algoritmo, funcion_obj, n_agents, n_indicios):
    """
    Modifica una configuración base con nuevos parámetros
    """
    config = copy.deepcopy(base_config)
    config["algoritmo_busqueda"] = algoritmo
    config["funcion_objetivo"] = funcion_obj if algoritmo in ["ACO", "ABC", "BHA"] else None
    config["num_agents"] = n_agents
    
    # Generar indicios aleatorios dentro del mapa
    size_x, size_y = config["size"]
    config["indicios"] = [
        [random.randint(0, size_x-5), random.randint(0, size_y-5)]
        for _ in range(n_indicios)
    ]
    
    # Configurar la carpeta de resultados con ruta absoluta
    # Usar str() para convertir Path a string, ya que JSON no soporta objetos Path
    config["carpeta_resultados"] = str(CARPETA_RESULTADOS.resolve()).replace("\\", "/")
    
    return config

def guardar_config(config, filename):
    """
    Guarda una configuración en formato JSON en la carpeta json/
    """
    ruta_completa = CARPETA_JSON / filename
    with open(ruta_completa, "w") as f:
        json.dump(config, f, indent=4)
    return str(ruta_completa)

def ejecutar_simulacion(ruta_config, n_iteraciones, simulador):
    """
    Ejecuta múltiples iteraciones de una simulación
    """
    # Cargar la configuración base UNA SOLA VEZ
    with open(ruta_config, "r") as f:
        base_config = json.load(f)
    
    for i in tqdm(range(n_iteraciones), desc="Iteraciones", unit="iter"):
        # Modificar solo la semilla
        config = copy.deepcopy(base_config)
        config["semilla"] = i
        
        # Sobrescribir el archivo de configuración con la nueva semilla
        with open(ruta_config, "w") as f:
            json.dump(config, f, indent=4)

        # Ejecutar el simulador
        result = subprocess.run(
            ["python3", simulador, ruta_config],
            capture_output=True, 
            text=True
        )
        
        # Opcional: mostrar errores si los hay
        if result.returncode != 0:
            print(f"\nError en iteración {i}:")
            print(result.stderr)

print("Funciones definidas")

Funciones definidas


## Simulación de pruebas

In [ ]:
for mapa in MAPAS:
    print(f"\n{'='*60}")
    print(f"Procesando mapa: {mapa}")
    print(f"{'='*60}\n")

    with open(mapa, "r") as f:
        config_base = json.load(f)

    # Calcular total de combinaciones
    total_combinaciones = (
        len(ALGORITMOS) * 
        len(NUM_AGENTES) * 
        len(NUM_INDICIOS) * 
        max(len(FUNCIONES_OBJ), 1)
    )
    
    with tqdm(total=total_combinaciones, desc="Combinaciones", unit="comb") as pbar:
        for algoritmo in ALGORITMOS:
            for n_agents in NUM_AGENTES:
                for n_indicios in NUM_INDICIOS:
                    # Solo usar funciones objetivo con algoritmos metaheurísticos
                    funciones = FUNCIONES_OBJ if algoritmo in ["ACO", "ABC", "BHA"] else [None]
                    
                    for funcion in funciones:
                        # Generar nombre corto del archivo de configuración
                        # Formato: mediano_ACO_ET_a1_i1.json (más corto para evitar problemas)
                        mapa_nombre = "med" if "mediano" in Path(mapa).stem else "gde"
                        filename = f"{mapa_nombre}_{algoritmo}"
                        if funcion:
                            filename += f"_{funcion}"
                        filename += f"_a{n_agents}_i{n_indicios}.json"
                        
                        # Crear y guardar configuración
                        config_mod = modificar_config(
                            config_base, algoritmo, funcion, n_agents, n_indicios
                        )
                        ruta_config = guardar_config(config_mod, filename)

                        # Ejecutar simulación
                        ejecutar_simulacion(ruta_config, N_ITERACIONES, alg)

                        # Notificar progreso
                        mensaje = (
                            f"Completado: {algoritmo} / FO={funcion} / "
                            f"agents={n_agents} / indicios={n_indicios}"
                        )
                        enviar_mensaje_telegram(mensaje)
                        print(mensaje)
                        
                        pbar.update(1)

enviar_mensaje_telegram("Todas las simulaciones han terminado.")
print("\n" + "="*60)
print("TODAS LAS SIMULACIONES HAN TERMINADO")
print("="*60)


Procesando mapa: pruebas/mediano-indicios-1-agentes-1-random.json



Combinaciones:   0%|          | 0/252 [00:16<?, ?comb/s]


KeyboardInterrupt: 